# Lab 12 — SOLUTION

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import KFold, StratifiedKFold, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression

energy = pd.read_csv("data/energy_demand.csv").sort_values("date").reset_index(drop=True)
tss = TimeSeriesSplit(n_splits=5)
for i, (tr, te) in enumerate(tss.split(energy)):
    print(f"fold {i}: train {energy.date.iloc[tr].min()}..{energy.date.iloc[tr].max()} -> test {energy.date.iloc[te].min()}..{energy.date.iloc[te].max()}")

fraud = pd.read_csv("data/fraud.csv")
Xf, yf = fraud.drop(columns="is_fraud"), fraud.is_fraud
for name, cv in [("KFold", KFold(5, shuffle=True, random_state=42)), ("StratifiedKFold", StratifiedKFold(5, shuffle=True, random_state=42))]:
    ratios = [yf[te].mean() for _, te in cv.split(Xf, yf)]
    print(name, "fold positive ratios:", np.round(ratios, 3))

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print("pipeline CV F1:", cross_val_score(pipe, Xf, yf, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1").round(3))

**Teaching notes:** the fold-date printout makes time-travel visible (train always ends before test begins). The KFold vs StratifiedKFold ratio table shows the imbalance hazard directly.